# Análise Exploratória — Pequenos Empresários e Autônomos

Este notebook analisa `Dados/clean_data.csv`, antes das agregações de histórico. O objetivo é entender o público selecionado, qualidade dos dados, desbalanceamento do TARGET e relações preliminares que orientam a construção da ABT.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name in {"DataPipeline", "Model"}:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from MLOps import storage

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. Carregamento e recorte do público

In [ ]:
df = storage.read_csv("Dados/clean_data.csv", low_memory=False)
print(f"Linhas: {len(df):,} | Colunas: {df.shape[1]}")
print(f"Taxa geral de default: {df['TARGET'].mean():.2%}")
if 'SMALL_BUSINESS_PROXY' in df:
    print(df['SMALL_BUSINESS_PROXY'].value_counts(dropna=False))
    small = df[df['SMALL_BUSINESS_PROXY']==1].copy()
else:
    small = df.copy()
print(f"Público analisado: {len(small):,} solicitações | Default: {small['TARGET'].mean():.2%}")

## 2. Distribuição do TARGET
Acurácia isolada é inadequada em base desbalanceada. A avaliação posterior prioriza AUC-ROC, KS e recall da classe inadimplente.

In [ ]:
target = small['TARGET'].value_counts().sort_index()
display(pd.DataFrame({'quantidade':target, 'percentual':target/target.sum()}))
ax=(target/target.sum()*100).plot(kind='bar', title='Distribuição do TARGET (%)')
ax.set_xlabel('TARGET (0=adimplente, 1=inadimplente)'); ax.set_ylabel('%'); plt.show()

## 3. Qualidade e valores ausentes

In [ ]:
missing=(small.isna().mean()*100).sort_values(ascending=False).to_frame('missing_pct')
display(missing.head(30))
print('Colunas com >50% de nulos:', int((missing.missing_pct>50).sum()))

## 4. Perfil financeiro e demográfico

In [ ]:
cols=[c for c in ['AMT_INCOME_TOTAL','AMT_CREDIT','AMT_ANNUITY','AMT_GOODS_PRICE','DAYS_BIRTH','DAYS_EMPLOYED','EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'] if c in small]
display(small[cols].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).T)

In [ ]:
if 'DAYS_BIRTH' in small:
    small['AGE_YEARS']=-small['DAYS_BIRTH']/365.25
    small['AGE_BUCKET']=pd.cut(small['AGE_YEARS'], bins=[18,30,40,50,60,100], right=False)
    age=small.groupby('AGE_BUCKET', observed=False)['TARGET'].agg(['count','mean'])
    display(age)
    age['mean'].mul(100).plot(marker='o', title='Taxa de inadimplência por faixa etária (%)'); plt.ylabel('%'); plt.show()

## 5. Razões financeiras
As razões capturam alavancagem e comprometimento de renda, reduzindo dependência dos valores absolutos.

In [ ]:
small['CREDIT_INCOME_RATIO']=small['AMT_CREDIT']/small['AMT_INCOME_TOTAL'].replace(0,np.nan)
small['ANNUITY_INCOME_RATIO']=small['AMT_ANNUITY']/small['AMT_INCOME_TOTAL'].replace(0,np.nan)
small['ANNUITY_CREDIT_RATIO']=small['AMT_ANNUITY']/small['AMT_CREDIT'].replace(0,np.nan)
ratio_cols=['CREDIT_INCOME_RATIO','ANNUITY_INCOME_RATIO','ANNUITY_CREDIT_RATIO']
display(small.groupby('TARGET')[ratio_cols].median().T)

## 6. Scores externos e associação com o TARGET

In [ ]:
ext=[c for c in ['EXT_SOURCE_1','EXT_SOURCE_2','EXT_SOURCE_3'] if c in small]
display(small.groupby('TARGET')[ext].agg(['mean','median','count']).T)
for c in ext:
    small.groupby('TARGET')[c].plot(kind='density', legend=True, title=f'{c} por TARGET')
    plt.xlim(0,1); plt.show()

## 7. Categóricas prioritárias

In [ ]:
categoricals=[c for c in ['NAME_INCOME_TYPE','ORGANIZATION_TYPE','NAME_EDUCATION_TYPE','CODE_GENDER','NAME_FAMILY_STATUS','NAME_CONTRACT_TYPE'] if c in small]
for c in categoricals:
    summary=small.groupby(c, dropna=False)['TARGET'].agg(['count','mean']).sort_values('count',ascending=False).head(15)
    summary['default_pct']=summary['mean']*100
    print(f"\n### {c}")
    display(summary[['count','default_pct']])

## 8. Conclusões para a ABT

- O recorte de pequenos empresários/autônomos precisa ser comparado ao universo geral em volume e taxa de default.
- O desbalanceamento exige métricas orientadas à separação e captura de inadimplentes.
- `EXT_SOURCE_*`, idade, renda, crédito e razões financeiras são candidatos centrais.
- Os dados comportamentais de `bureau`, `previous_application` e `installments_payments` serão agregados no próximo estágio.
- Colunas extremamente nulas devem ser descartadas, preservando scores externos pelo sinal preditivo.